In [1]:
import polars as pl
import numpy as np
import pandas as pd
import optuna
import pickle
import os
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [6]:
# ------------------------------
# 1. Объединение main и extra признаков
# ------------------------------
def build_wide_data():
    """Загружает и объединяет main и extra признаки, приводит к float16"""
    print("Загрузка и объединение признаков...")    
    # Загружаем customer_id отдельно
    train_customer_ids = pl.read_parquet('data/train_main_features.parquet').select('customer_id')
    test_customer_ids = pl.read_parquet('data/test_main_features.parquet').select('customer_id')
    
    # Сканируем main и extra (используем lazy для экономии памяти)
    lf_main_train = pl.scan_parquet('data/train_main_features.parquet').drop('customer_id')
    lf_extra_train = pl.scan_parquet('data/train_extra_features.parquet').drop('customer_id')
    lf_main_test = pl.scan_parquet('data/test_main_features.parquet').drop('customer_id')
    lf_extra_test = pl.scan_parquet('data/test_extra_features.parquet').drop('customer_id')
    
    # Объединяем по горизонтали и приводим к float16 для экономии памяти
    train_wide = pl.concat([lf_main_train, lf_extra_train], how='horizontal').with_columns(
        pl.all().cast(pl.Float16)
    ).collect()
    
    test_wide = pl.concat([lf_main_test, lf_extra_test], how='horizontal').with_columns(
        pl.all().cast(pl.Float16)
    ).collect()
    
    print(f"Train wide shape: {train_wide.shape}")
    print(f"Test wide shape: {test_wide.shape}")
    return train_wide, test_wide, train_customer_ids, test_customer_ids

train_wide, test_wide, train_customer_ids, test_customer_ids = build_wide_data()

Загрузка и объединение признаков...
Train wide shape: (750000, 2440)
Test wide shape: (250000, 2440)


In [7]:
# ------------------------------
# 2. Удаление дубликатов столбцов
# ------------------------------
def remove_duplicate_columns(df):
    """Удаляет дубликаты столбцов (оставляет первые вхождения)"""
    unique_cols = []
    seen = set()
    for col in df.columns:
        if col not in seen:
            unique_cols.append(col)
            seen.add(col)
    return df.select(unique_cols)

train_wide = remove_duplicate_columns(train_wide)
test_wide = remove_duplicate_columns(test_wide)

print(f"После удаления дубликатов: train shape {train_wide.shape}, test shape {test_wide.shape}")

После удаления дубликатов: train shape (750000, 2440), test shape (250000, 2440)


In [9]:
# ------------------------------
# 3. Загрузка целевых переменных и предобработка
# ------------------------------
print("\nЗагрузка целевых переменных...")
target = pl.read_parquet('data/train_target.parquet')
target_cols = [col for col in target.columns if col.startswith("target")]
y_all = target.select(target_cols).to_pandas()

# Определяем категориальные признаки (по префиксу "cat_feature")
cat_features = [col for col in train_wide.columns if col.startswith("cat_feature")]
print(f"Категориальных признаков: {len(cat_features)}")

# Исправленная функция заполнения пропусков
def fill_missing(df, cat_features):
    """Заполняет пропуски: числовые медианой, категориальные -1"""
    df = df.clone()
    
    # Числовые колонки (все, кроме категориальных)
    num_cols = [col for col in df.columns if col not in cat_features]
    
    for col in num_cols:
        # Получаем медиану, игнорируя NaN
        median_val = df[col].drop_nulls().median()
        
        # Если медиана не None (т.е. есть хотя бы одно не-NaN значение)
        if median_val is not None:
            df = df.with_columns(pl.col(col).fill_null(median_val))
        else:
            # Если все значения NaN, заполняем 0
            df = df.with_columns(pl.col(col).fill_null(0))
    
    # Заполнение категориальных признаков
    for col in cat_features:
        df = df.with_columns(pl.col(col).fill_null(-1).cast(pl.Int32))
    
    return df

print("\nЗаполнение пропусков...")
train_wide = fill_missing(train_wide, cat_features)
test_wide = fill_missing(test_wide, cat_features)


Загрузка целевых переменных...
Категориальных признаков: 67

Заполнение пропусков...


In [10]:
# ------------------------------
# 4. Подготовка данных для обучения
# ------------------------------
X = train_wide.to_pandas()
X_test = test_wide.to_pandas()

# Убедимся, что все категориальные признаки имеют тип int
for col in cat_features:
    if col in X.columns:
        X[col] = X[col].fillna(-1).astype(int)
    if col in X_test.columns:
        X_test[col] = X_test[col].fillna(-1).astype(int)

# Стратифицированное разделение на train/val (80/20)
print("\nРазделение на train/val...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y_all, test_size=0.2, random_state=42,
    stratify=y_all['target_1_1']  # стратификация по первой метке
)


Разделение на train/val...


In [17]:
# ------------------------------
# 5. Ручные параметры для каждого класса (на основе успешных оптимизаций)
# ------------------------------
def get_params_for_target(target_name, val_auc=None):
    """Возвращает оптимальные параметры для каждого класса на основе успешных оптимизаций"""
    
    # Группируем классы по их характеристикам на основе валидационного AUC из предыдущих запусков
    # Используем логику: для сложных классов (низкий AUC) - более глубокие деревья и больше итераций
    # Для простых классов (высокий AUC) - более простые модели
    
    # Извлечем номер группы из target_name (например, target_1_1 -> группа 1)
    group = int(target_name.split('_')[1])
    
    # Базовые параметры для всех классов
    base_params = {
        'learning_rate': 0.05,
        'l2_leaf_reg': 4,
        'random_strength': 2,
        'bagging_temperature': 0.8,
        'border_count': 128,
        'min_data_in_leaf': 10,
        'early_stopping_rounds': 50,
        'eval_metric': 'AUC',
        'loss_function': 'Logloss',
        'verbose': False,
        'random_seed': 42,
        'task_type': 'CPU',
        'thread_count': 4
    }
    
    # Специфические параметры для разных групп
    if target_name in ['target_2_8', 'target_3_5', 'target_8_1']:
        # Очень простые классы (AUC > 0.96)
        params = {
            'iterations': 200,
            'depth': 5,
            'learning_rate': 0.08,
            'l2_leaf_reg': 5,
            'random_strength': 1,
            'bagging_temperature': 0.6,
            'min_data_in_leaf': 15
        }
    
    elif target_name in ['target_2_2', 'target_3_4']:
        # Хорошо предсказуемые классы (AUC 0.91-0.93)
        params = {
            'iterations': 250,
            'depth': 4,
            'learning_rate': 0.07,
            'l2_leaf_reg': 4,
            'random_strength': 2,
            'bagging_temperature': 0.7,
            'min_data_in_leaf': 10
        }
    
    elif target_name in ['target_1_1', 'target_1_3', 'target_1_5']:
        # Средние классы (AUC 0.84-0.89)
        params = {
            'iterations': 300,
            'depth': 6,
            'learning_rate': 0.06,
            'l2_leaf_reg': 4,
            'random_strength': 2,
            'bagging_temperature': 0.7,
            'min_data_in_leaf': 12
        }
    
    elif target_name in ['target_3_1', 'target_2_4', 'target_9_3', 'target_9_6']:
        # Сложные классы (AUC < 0.68) - больше итераций, глубже
        params = {
            'iterations': 500,
            'depth': 6,
            'learning_rate': 0.05,
            'l2_leaf_reg': 3,
            'random_strength': 3,
            'bagging_temperature': 0.9,
            'min_data_in_leaf': 8
        }
    
    elif target_name in ['target_1_2', 'target_2_6', 'target_6_3', 'target_9_1', 'target_9_4']:
        # Классы, где depth=3 показал хороший результат
        params = {
            'iterations': 350,
            'depth': 3,
            'learning_rate': 0.07,
            'l2_leaf_reg': 4,
            'random_strength': 3,
            'bagging_temperature': 0.55,
            'min_data_in_leaf': 6
        }
    
    elif target_name in ['target_2_5', 'target_2_7', 'target_4_1', 'target_6_5']:
        # Классы с iterations=100, depth=6
        params = {
            'iterations': 400,
            'depth': 6,
            'learning_rate': 0.07,
            'l2_leaf_reg': 6,
            'random_strength': 2,
            'bagging_temperature': 0.6,
            'min_data_in_leaf': 12
        }
    
    elif target_name in ['target_5_2']:
        # Специфичные параметры
        params = {
            'iterations': 450,
            'depth': 4,
            'learning_rate': 0.04,
            'l2_leaf_reg': 6,
            'random_strength': 2,
            'bagging_temperature': 0.9,
            'min_data_in_leaf': 14
        }
    
    elif target_name in ['target_9_5', 'target_9_8']:
        # Классы с iterations=300, depth=3
        params = {
            'iterations': 450,
            'depth': 3,
            'learning_rate': 0.06,
            'l2_leaf_reg': 6,
            'random_strength': 4,
            'bagging_temperature': 0.65,
            'min_data_in_leaf': 6
        }
    
    else:
        # Дефолтные параметры для остальных классов
        params = {
            'iterations': 350,
            'depth': 5,
            'learning_rate': 0.06,
            'l2_leaf_reg': 4,
            'random_strength': 2,
            'bagging_temperature': 0.7,
            'min_data_in_leaf': 10
        }
    
    # Объединяем с базовыми параметрами
    params.update(base_params)
    return params


In [ ]:
# ------------------------------
# 4. Разбиение на группы
# ------------------------------
num_groups = 8
group_size = len(target_cols) // num_groups
groups = [target_cols[i*group_size:(i+1)*group_size] for i in range(num_groups)]
# Добавляем остаток в последнюю группу
if len(groups[-1]) < len(target_cols) - (num_groups-1)*group_size:
    groups[-1].extend(target_cols[num_groups*group_size:])
print("Группы целевых переменных:")
for i, g in enumerate(groups):
    print(f"  Группа {i+1}: {len(g)} классов")

Группы целевых переменных:
  Группа 1: 5 классов
  Группа 2: 5 классов
  Группа 3: 5 классов
  Группа 4: 5 классов
  Группа 5: 5 классов
  Группа 6: 5 классов
  Группа 7: 5 классов
  Группа 8: 6 классов


In [19]:
# ------------------------------
# 6. Обучение моделей с ручными параметрами
# ------------------------------
models = {}
val_auc = {}

In [20]:
def study(target_name):
    print(f"\nОбучаем {target_name}...")
    # Получаем параметры для текущего класса
    params = get_params_for_target(target_name)
    
    # Веса классов для борьбы с дисбалансом
    pos = y_train[target_name].sum()
    neg = len(y_train) - pos
    if pos > 0 and neg > 0:
        params['class_weights'] = {0: 1, 1: neg / pos}
    
    print(f"  Параметры: iterations={params['iterations']}, depth={params['depth']}, lr={params['learning_rate']:.3f}")
    
    # Создаем и обучаем модель
    model = CatBoostClassifier(**params)
    train_pool = Pool(X_train, label=y_train[target_name], cat_features=cat_features)
    val_pool = Pool(X_val, label=y_val[target_name], cat_features=cat_features)
    
    model.fit(train_pool, eval_set=val_pool)
    
    # Оценка
    pred_val = model.predict_proba(val_pool)[:, 1]
    auc = roc_auc_score(y_val[target_name], pred_val)
    val_auc[target_name] = auc
    models[target_name] = model
    print(f"  ROC-AUC: {auc:.4f}")

In [21]:
#group 1
for target_name in groups[0]:
    study(target_name)


Обучаем target_1_1...
  Параметры: iterations=300, depth=6, lr=0.050


KeyboardInterrupt: 

In [ ]:
#group 2
for target_name in groups[1]:
    study(target_name)

[I 2026-03-21 22:36:01,905] A new study created in memory with name: no-name-f5eafa0f-3b35-4e02-8e95-24ed96377f82



Оптимизация для target_2_6...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 22:36:34,124] Trial 0 finished with value: 0.6925373789407423 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.6925373789407423.
[I 2026-03-21 22:36:56,400] Trial 1 finished with value: 0.6350845176976074 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.6925373789407423.
[I 2026-03-21 22:37:29,525] Trial 2 finished with value: 0.6321697399499882 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.69253

[I 2026-03-21 22:38:26,969] A new study created in memory with name: no-name-66f25b4f-a6f8-4a7e-b830-a725c795f3a7


...

[I 2026-03-21 22:52:43,964] Trial 0 finished with value: 0.8519561913872464 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.8519561913872464.
[I 2026-03-21 22:53:30,891] Trial 1 finished with value: 0.8521661201272391 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 1 with value: 0.8521661201272391.
[I 2026-03-21 22:54:00,336] Trial 2 finished with value: 0.8442844041722597 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 1 with value: 0.85216

[I 2026-03-21 22:54:45,473] A new study created in memory with name: no-name-226c6a77-2cd1-4e90-8f1a-0212c31b1bc1


[I 2026-03-21 22:54:45,471] Trial 5 finished with value: 0.8537470091860413 and parameters: {'iterations': 100, 'learning_rate': 0.04437555550097432, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 2, 'bagging_temperature': 0.9637655990477874, 'border_count': 64, 'min_data_in_leaf': 16}. Best is trial 4 with value: 0.8569135167002759.
Лучший AUC: 0.8569, параметры: {'iterations': 100, 'learning_rate': 0.09210273435223537, 'depth': 6, 'l2_leaf_reg': 6, 'random_strength': 2, 'bagging_temperature': 0.5683704798044688, 'border_count': 96, 'min_data_in_leaf': 14}

Оптимизация для target_5_1...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 22:55:15,279] Trial 0 finished with value: 0.7296898141914636 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.7296898141914636.
[I 2026-03-21 22:55:40,878] Trial 1 finished with value: 0.7198717056057959 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.7296898141914636.
[I 2026-03-21 22:56:16,843] Trial 2 finished with value: 0.7250318050024809 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.72968

In [ ]:
#group 3
for target_name in groups[2]:
    study(target_name)

[I 2026-03-21 22:57:15,326] A new study created in memory with name: no-name-1e5a9b72-302d-4c89-a515-cc0b41dc6e3a



Оптимизация для target_5_2...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 22:57:43,896] Trial 0 finished with value: 0.6827209899195001 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.6827209899195001.
[I 2026-03-21 22:57:56,647] Trial 1 finished with value: 0.6687506116319407 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.6827209899195001.
[I 2026-03-21 22:58:04,831] Trial 2 finished with value: 0.66111039740068 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.6827209

[I 2026-03-21 22:58:26,941] A new study created in memory with name: no-name-c93508b4-a38b-4515-95c8-efb62b480c6f


...

[I 2026-03-21 23:17:28,207] Trial 0 finished with value: 0.7151739476025145 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.7151739476025145.
[I 2026-03-21 23:17:35,970] Trial 1 finished with value: 0.6647827910149772 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.7151739476025145.
[I 2026-03-21 23:17:47,468] Trial 2 finished with value: 0.6674665586900874 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.71517

[I 2026-03-21 23:18:51,887] A new study created in memory with name: no-name-6d029f9f-3a84-44e2-9ca8-b02899f5e6aa


[I 2026-03-21 23:18:51,885] Trial 5 finished with value: 0.6907577163933101 and parameters: {'iterations': 100, 'learning_rate': 0.04437555550097432, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 2, 'bagging_temperature': 0.9637655990477874, 'border_count': 64, 'min_data_in_leaf': 16}. Best is trial 3 with value: 0.7426294691704824.
Лучший AUC: 0.7426, параметры: {'iterations': 250, 'learning_rate': 0.07076922519051033, 'depth': 3, 'l2_leaf_reg': 4, 'random_strength': 3, 'bagging_temperature': 0.5325152889039984, 'border_count': 96, 'min_data_in_leaf': 6}

Оптимизация для target_8_1...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 23:19:53,229] Trial 0 finished with value: 0.9637273766257822 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.9637273766257822.
[I 2026-03-21 23:20:44,968] Trial 1 finished with value: 0.9611029073003421 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.9637273766257822.
[I 2026-03-21 23:21:30,302] Trial 2 finished with value: 0.9559356851789061 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.96372

In [ ]:
#group 4
for target_name in groups[3]:
    study(target_name)

[I 2026-03-21 23:27:38,547] A new study created in memory with name: no-name-5780cc65-2b54-411d-b97b-41e185d16063



Оптимизация для target_8_2...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 23:28:40,168] Trial 0 finished with value: 0.8168028471094473 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.8168028471094473.
[I 2026-03-21 23:28:59,735] Trial 1 finished with value: 0.7409589722874033 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.8168028471094473.
[I 2026-03-21 23:29:40,920] Trial 2 finished with value: 0.7635410042234279 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.81680

[I 2026-03-21 23:30:49,180] A new study created in memory with name: no-name-3b07613b-88ff-4fd3-ac50-2bd4d8269929


...

[I 2026-03-21 23:54:26,140] Trial 0 finished with value: 0.9221648084515299 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.9221648084515299.
[I 2026-03-21 23:55:15,260] Trial 1 finished with value: 0.9232495338922747 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 1 with value: 0.9232495338922747.
[I 2026-03-21 23:55:53,274] Trial 2 finished with value: 0.908874609538125 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 1 with value: 0.923249

[I 2026-03-21 23:56:57,677] A new study created in memory with name: no-name-a0c093e0-7c73-4afa-a9aa-8348dc67bfb9


[I 2026-03-21 23:56:57,675] Trial 5 finished with value: 0.9112279475874829 and parameters: {'iterations': 100, 'learning_rate': 0.04437555550097432, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 2, 'bagging_temperature': 0.9637655990477874, 'border_count': 64, 'min_data_in_leaf': 16}. Best is trial 1 with value: 0.9232495338922747.
Лучший AUC: 0.9232, параметры: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}

Оптимизация для target_10_1...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 23:57:56,229] Trial 0 finished with value: 0.7386493822511297 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.7386493822511297.
[I 2026-03-21 23:58:46,353] Trial 1 finished with value: 0.7327357845932962 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.7386493822511297.
[I 2026-03-21 23:59:29,345] Trial 2 finished with value: 0.7277264794471681 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.73864

In [ ]:
#group 5
for target_name in groups[4]:
    study(target_name)

{'target_1_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_2': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_1_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_4': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_5': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l

In [ ]:
#group 6
for target_name in groups[5]:
    study(target_name)


Обучение финальных моделей на всех данных...


In [ ]:
#group 7
for target_name in groups[6]:
    study(target_name)

Обучаем target_1_1...
Обучаем target_1_2...
Обучаем target_1_3...
Обучаем target_1_4...
Обучаем target_1_5...
Обучаем target_2_1...
Обучаем target_2_2...
Обучаем target_2_3...
Обучаем target_2_4...
Обучаем target_2_5...


In [ ]:
#group 8
for target_name in groups[7]:
    study(target_name)

Обучаем target_2_6...
Обучаем target_2_7...
Обучаем target_2_8...
Обучаем target_3_1...
Обучаем target_3_2...
Обучаем target_3_3...
Обучаем target_3_4...
Обучаем target_3_5...
Обучаем target_4_1...
Обучаем target_5_1...


In [ ]:
# ------------------------------
# 7. Предсказание на тестовых данных
# ------------------------------
print("\nФормирование предсказаний...")
test_pool = Pool(X_test, cat_features=cat_features)

predictions = []
for target_name in target_cols:
    model = models[target_name]
    proba = model.predict_proba(test_pool)[:, 1]
    predictions.append(proba)

predictions = np.column_stack(predictions)
predict_cols = [f"predict_{col.replace('target_', '')}" for col in target_cols]
pred_df = pl.DataFrame(predictions, schema=predict_cols)


Формирование предсказаний...


In [ ]:
# ------------------------------
# 8. Сохранение сабмита
# ------------------------------
submit = pl.DataFrame({'customer_id': test_customer_ids.to_pandas()['customer_id']}).hstack(pred_df)
submit.write_parquet("data/submit3.parquet")
print("\nСабмит сохранён в data/submit3.parquet")
print(f"\nСредний ROC-AUC на валидации: {np.mean(list(val_auc.values())):.4f}")

# Статистика по группам
print("\nСтатистика по AUC для классов:")
for target_name, auc in sorted(val_auc.items(), key=lambda x: x[1], reverse=True):
    print(f"  {target_name}: {auc:.4f}")

# Вывод среднего AUC по группам продуктов
print("\nСредний AUC по группам продуктов:")
for group in range(1, 11):
    group_targets = [t for t in target_cols if t.startswith(f"target_{group}_")]
    if group_targets:
        group_auc = np.mean([val_auc[t] for t in group_targets])
        print(f"  Группа {group}: {group_auc:.4f}")